# Prepare LIAISE stations datas

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import os

import geopandas as gpd
from rasterio.coords import BoundingBox

In [ ]:
from temperature.liaise import (
    get_method_paths,
    read_netcdf_liaise_data,
    save_liaise_station,
    station_els_plans_to_csv,
    station_irta_corn_to_csv,
)
from temperature.plot import plot_liaise_data

from etdataset.dem import get_dem_from_roi
from etdataset.era5 import ERA5Dataset
from etdataset.plot import plot_dataset, plot_dem
from etdataset.temperature import (
    RescalTempMethod,
    TempVariable,
    add_temp,
)


In [ ]:
import logging

from etdataset.logging import LoggerManager

LoggerManager.set_level(logging.DEBUG)

In [ ]:
gdf = gpd.read_file("stage_liaise/Zone_Clip/Zone_etude.shp")

In [ ]:
xmin, ymin, xmax, ymax = gdf.total_bounds

In [ ]:
roi_bbox_utm = BoundingBox(left=xmin, bottom=ymin, right=xmax, top=ymax)
roi_crs_utm = gdf.crs

In [ ]:
date = dt.datetime(2026, 4, 1, 12, 0)

In [ ]:
mnt_path = os.environ["MNT_PATH"]

In [ ]:
data = get_dem_from_roi(
    roi_bbox=roi_bbox_utm,
    roi_crs=roi_crs_utm,
    base_dir=mnt_path,
    resolution=60,
)

In [ ]:
# Add attributes
data.attrs["vis_date"] = date.date()
data.attrs["vis_time"] = date.time()
data.attrs["tir_date"] = date.date()
data.attrs["tir_time"] = date.time()
data


In [ ]:
plot_dem(data)

In [ ]:
updated_data = add_temp(
    data=data,
    dataset=ERA5Dataset.ERA5,
    variables=[TempVariable.TD, TempVariable.TA],
    method=RescalTempMethod.CONST_LR,
)

In [ ]:
plot_dataset(updated_data)

In [ ]:
updated_data

In [ ]:
get_method_paths("data/LIAISE/stations", 2021, 5, 3)

In [ ]:
plot_liaise_data("data/LIAISE/stations", 2021, 10, 1, "td")

In [ ]:
df = read_netcdf_liaise_data("data/LIAISE/PREIXANA/V2")

In [ ]:
save_liaise_station(
    df,
    "PREIXANA",
    "data/LIAISE/liaise_processed/",
)

In [ ]:
station_irta_corn_to_csv(
    "data/LIAISE/IRTA-CORN/LIAISE_IRTA-CORN_UIB_SEB-10MIN_L2/LIAISE_IRTA-CORN_UIB_SEB1-10MIN_L2.dat",
    "data/LIAISE/liaise_processed/IRTA-CORN_processed.csv",
)


In [ ]:
station_els_plans_to_csv(
    folder="data/LIAISE/ELS-PLANS/LIAISE_ELS-PLANS_UKMO_MTO-30MIN_L2",
    output_path="data/LIAISE/liaise_processed/ELS-PLANS_processed.csv",
)